## Week 12 Practice

In [62]:
# Deep Neural Network 
import torch
import torch.nn as nn 
import torch.nn.functional as F
import torch.optim as optim

# Define model: two-layer MLP with linear layer 
class MultilayerPerceptron(nn.Module): 
    def __init__(self, num_features, hidden_size1, hidden_size2, num_classes, drop_proba):
        super(MultilayerPerceptron, self).__init__()

        # dropout layer
        self.drop_proba = drop_proba

        # hidden layers
        self.linear_1 = nn.Linear(num_features, hidden_size1)
        self.linear_2 = nn.Linear(hidden_size1, hidden_size2)

        # output layer 
        self.linear_out = nn.Linear(hidden_size2, num_classes)

    def forward(self, x): 
        x = F.relu(self.linear_1(x))
        x = F.dropout(x, p=self.drop_proba, training=self.training)  # dropout for regularization
        x = F.relu(self.linear_2(x))
        x = F.dropout(x, p=self.drop_proba, training=self.training)  # dropout for regularization
        logits = self.linear_out(x)
        probas = torch.sigmoid(logits)
        return logits, probas

# # equivalently: 
#     # more compact; harder to debug if there are errors 
# class MultilayerPerceptron(torch.nn.Module): 
#     def __init__(self, num_features, num_classes): 
#         super(MultilayerPerceptron, self).__init__()

#         self.my_network = torch.nn.Sequential(
#             torch.nn.Linear(num_features, hidden_size1),
#             torch.nn.ReLU(),
#             torch.nn.Linear(hidden_size1, hidden_size2),
#             torch.nn.ReLU(),
#             torch.nn.Linear(hidden_size2, num_classes),
#             torch.nn.Sigmoid()
#         )
    
#     def forward(self, x):
#         logits = self.my_network(x)
#         probas = F.softmax(logits, dim=1)
#         return logits, probas

In [65]:
# initialize neural network model
num_features = torch.randn(10).shape[0]  # example feature size (will be overridden)
hidden_size1 = 64
hidden_size2 = 32
num_features = 2  # binary classification (2 input features)
num_classes = 1   # single output for binary classification
learning_rate = 0.01
# model = MultilayerPerceptron(num_features, num_classes); print(model)
model = MultilayerPerceptron(num_features, hidden_size1, hidden_size2, num_classes, drop_proba=0.2)
print(model)

# define loss function (binary cross-entropy) and optimizer (SGD)
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

MultilayerPerceptron(
  (linear_1): Linear(in_features=2, out_features=64, bias=True)
  (linear_2): Linear(in_features=64, out_features=32, bias=True)
  (linear_out): Linear(in_features=32, out_features=1, bias=True)
)


In [66]:
# Training 
X = torch.randn(100, num_features)  # example input data
y = torch.randint(0, 2, (100, 1)).float()  # example binary target data
epochs = 1000
for epoch in range(epochs): 
    # forward pass 
    logits, outputs = model(X) 

    # compute loss 
    loss = criterion(outputs, y)

    # Backpropogation and optimization
    optimizer.zero_grad() # clear gradients
    loss.backward()
    optimizer.step() # update weights

    if (epoch + 1) % 100 == 0: 
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')


Epoch [100/1000], Loss: 0.7002
Epoch [200/1000], Loss: 0.6944
Epoch [300/1000], Loss: 0.6927
Epoch [400/1000], Loss: 0.6901
Epoch [300/1000], Loss: 0.6927
Epoch [400/1000], Loss: 0.6901
Epoch [500/1000], Loss: 0.6922
Epoch [600/1000], Loss: 0.6869
Epoch [500/1000], Loss: 0.6922
Epoch [600/1000], Loss: 0.6869
Epoch [700/1000], Loss: 0.6781
Epoch [700/1000], Loss: 0.6781
Epoch [800/1000], Loss: 0.6781
Epoch [800/1000], Loss: 0.6781
Epoch [900/1000], Loss: 0.6828
Epoch [1000/1000], Loss: 0.6808
Epoch [900/1000], Loss: 0.6828
Epoch [1000/1000], Loss: 0.6808


### Loading and Batching Data in PyTorch 

In [67]:
print(model)

MultilayerPerceptron(
  (linear_1): Linear(in_features=2, out_features=64, bias=True)
  (linear_2): Linear(in_features=64, out_features=32, bias=True)
  (linear_out): Linear(in_features=32, out_features=1, bias=True)
)


In [68]:
# Loading and Batching Data in PyTorch 
from torch.utils.data import TensorDataset, DataLoader 
import numpy as np

np.random.seed(0)
X = np.random.rand(100, 2)
y = (X[:, 0] + X[:, 1] > 1).astype(int)

# Convert data to PyTorch tensors 
X = torch.tensor(X, dtype = torch.float32)
y = torch.tensor(y, dtype = torch.float32).unsqueeze(1) # reshape to (100, 1) 

dataset = TensorDataset(X, y)
batch_size = 32
dataLoader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Reinitialize model to ensure input feature size matches DataLoader
num_features = X.shape[1]  # should be 2
num_classes = 1
model = MultilayerPerceptron(num_features, hidden_size1, hidden_size2, num_classes, drop_proba=0.2)
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr=0.05)
print('Reinitialized model for DataLoader batches:', model)

Reinitialized model for DataLoader batches: MultilayerPerceptron(
  (linear_1): Linear(in_features=2, out_features=64, bias=True)
  (linear_2): Linear(in_features=64, out_features=32, bias=True)
  (linear_out): Linear(in_features=32, out_features=1, bias=True)
)


In [69]:
# checking attributes of DataLoader
for attr, value in dataLoader.__dict__.items():
    print(f"{attr}: {value}")

dataset: <torch.utils.data.dataset.TensorDataset object at 0x00000176A780D9D0>
num_workers: 0
prefetch_factor: None
pin_memory: False
pin_memory_device: 
timeout: 0
worker_init_fn: None
_DataLoader__multiprocessing_context: None
in_order: True
_dataset_kind: 0
batch_size: 32
drop_last: False
sampler: <torch.utils.data.sampler.RandomSampler object at 0x00000176A780DAE0>
batch_sampler: <torch.utils.data.sampler.BatchSampler object at 0x00000176A780DBF0>
generator: None
collate_fn: <function default_collate at 0x00000176A08C25C0>
persistent_workers: False
_DataLoader__initialized: True
_IterableDataset_len_called: None
_iterator: None


In [70]:
for batch_idx, (data, labels) in enumerate(dataLoader):
    # data: tensor of shape (batch_size, num_features)
    # labels: tensor of shape (batch_size)
    # training code here
    print(batch_idx, data, labels)

0 tensor([[0.1355, 0.2983],
        [0.5218, 0.4147],
        [0.4376, 0.8918],
        [0.8062, 0.7039],
        [0.4359, 0.8919],
        [0.7221, 0.8664],
        [0.4237, 0.6459],
        [0.1862, 0.9444],
        [0.6531, 0.2533],
        [0.1238, 0.8480],
        [0.6668, 0.6706],
        [0.6028, 0.5449],
        [0.7300, 0.1716],
        [0.0188, 0.6176],
        [0.7937, 0.2239],
        [0.8817, 0.6925],
        [0.9437, 0.6818],
        [0.6602, 0.2901],
        [0.8073, 0.5691],
        [0.6974, 0.4535],
        [0.2894, 0.1832],
        [0.1002, 0.9195],
        [0.6121, 0.6169],
        [0.3595, 0.4370],
        [0.3982, 0.2098],
        [0.8289, 0.0047],
        [0.5666, 0.2654],
        [0.0192, 0.3016],
        [0.3186, 0.6674],
        [0.6778, 0.2700],
        [0.3180, 0.4143],
        [0.9768, 0.6048]]) tensor([[0.],
        [0.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [1.],
        [0.],
        [0.],
        [1.],
        [1.

In [71]:
epochs = 50
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for batch_idx, (batch_X, batch_y) in enumerate(dataLoader):
        optimizer.zero_grad()
        logits, probas = model(batch_X)  # unpack logits and probabilities
        loss = criterion(probas, batch_y)  # batch_y already (batch,1)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        avg_loss = running_loss / len(dataLoader)
        with torch.no_grad():
            _, preds = model(X)
            acc = ((preds > 0.5).float() == y).float().mean().item()
        print(f"Epoch {epoch+1}: loss={avg_loss:.4f} acc={acc:.3f}")

Epoch 10: loss=0.6429 acc=0.520
Epoch 20: loss=0.6913 acc=0.530
Epoch 30: loss=0.6352 acc=0.620
Epoch 40: loss=0.5918 acc=0.710
Epoch 50: loss=0.4939 acc=0.740
Epoch 40: loss=0.5918 acc=0.710
Epoch 50: loss=0.4939 acc=0.740
